In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

def process(df):
    df['Deck'] = df['Cabin'].str.split('/').str[0]
    df['Side'] = df['Cabin'].str.split('/').str[2]
    df['Group'] = df['PassengerId'].str.split('_').str[0]
    df['GroupSize'] = df.groupby('Group')['PassengerId'].transform('count')
    df['Alone'] = (df['GroupSize'] == 1).astype(int)

    df['TotalSpend'] = df[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)
    df['ZeroSpender'] = (df['TotalSpend'] == 0).astype(int)

    df['TotalSpend'] = np.log1p(df['TotalSpend'])

    df['CryoSleep'] = df['CryoSleep'].fillna(df['ZeroSpender'].map({1: True, 0: np.nan}))
    df['CryoSleep'] = df['CryoSleep'].fillna(False)

    df['VIP'] = df['VIP'].fillna(False)
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['HomePlanet'] = df['HomePlanet'].fillna('Earth')
    df['Destination'] = df['Destination'].fillna('TRAPPIST-1e')
    df['Side'] = df['Side'].fillna('P')
    df['Deck'] = df['Deck'].fillna('F')

    return df


train = process(train)
test = process(test)

X = train.drop(columns=['PassengerId', 'Name', 'Cabin', 'Transported', 'Group'])
y = train['Transported'].astype(int)
X_test = test.drop(columns=['PassengerId', 'Name', 'Cabin', 'Group'])

categorical = X.select_dtypes(include="object").columns.tolist()
numerical = X.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical),
    ('num', numerical_pipeline, numerical)
])

# Base Models (with best hyper-parameters)
xgb = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    n_estimators=465,
    learning_rate=0.04974445885593547,
    max_depth=4,
    subsample=0.7023223983232721,
    colsample_bytree=0.8807540583107164,
    min_child_weight=1,
    gamma=0.3091801139027462
)

lgb = LGBMClassifier(
    random_state=42,
    n_estimators=337,
    learning_rate=0.024230875709953033,
    max_depth=9,
    num_leaves=26,
    subsample=0.9242197257138703,
    colsample_bytree=0.6030490088823314,
    min_child_samples=18,
    verbose=-1,
    silent=True,
)

rf = RandomForestClassifier(
    random_state=42,
    n_estimators=192,
    max_depth=11,
    min_samples_split=7,
    min_samples_leaf=7,
    max_features='sqrt'
)

mlp = MLPClassifier(
    random_state=42,
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0017466743092964523,
    learning_rate_init=0.0001985876989455488,
    max_iter=500
)
calibrated_mlp = CalibratedClassifierCV(mlp, method='sigmoid', cv=3)

models = {
    "xgb": xgb,
    "lgb": lgb,
    "rf": rf,
    "mlp": calibrated_mlp
}


# CV and prediction storage
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros((X_test.shape[0], len(models)))
val_scores = {name: [] for name in models}
best_thresholds = {name: [] for name in models}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    X_train_prep = preprocessor.fit_transform(X_train_fold)
    X_val_prep = preprocessor.transform(X_val_fold)
    X_test_prep = preprocessor.transform(X_test)

    for i, (name, model) in enumerate(models.items()):
        model.fit(X_train_prep, y_train_fold)
        val_pred_proba = model.predict_proba(X_val_prep)[:, 1]

        # Simple threshold optimization
        best_threshold = 0.5
        best_score = 0
        for threshold in np.arange(0.3, 0.7, 0.01):
            val_pred = (val_pred_proba >= threshold).astype(int)
            score = accuracy_score(y_val_fold, val_pred)
            if score > best_score:
                best_score = score
                best_threshold = threshold

        best_thresholds[name].append(best_threshold)
        val_scores[name].append(best_score)
        print(f"Fold {fold+1} | {name} | Val Accuracy: {best_score:.4f} | Threshold: {best_threshold:.3f}")

        test_preds[:, i] += model.predict_proba(X_test_prep)[:, 1] / kf.n_splits

# Print results
for name in models:
    print(f"{name} mean CV accuracy: {np.mean(val_scores[name]):.4f}")
    print(f"{name} mean optimal threshold: {np.mean(best_thresholds[name]):.3f}")

# Simple weighted ensemble using CV scores
weights = {name: np.mean(scores) for name, scores in val_scores.items()}
weight_sum = sum(weights.values())
weights = {k: v/weight_sum for k, v in weights.items()}

# Final prediction
final_preds = np.zeros(X_test.shape[0])
for i, (name, _) in enumerate(models.items()):
    mean_threshold = np.mean(best_thresholds[name])
    final_preds += weights[name] * (test_preds[:, i] >= mean_threshold).astype(int)

final_pred = (final_preds >= 0.5).astype(bool)

# Prepare submission
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': final_pred
})
submission.to_csv("submission_tuned.csv", index=False)
print("Submission file created!")